# Brouillon Julien — partie 2A (données) et 2B (modèles)

Le **test n'est pas ouvert** ici : tout se compare sur la validation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

users = pd.read_csv('rev-users.csv', parse_dates=['created_date'])
notifs = pd.read_csv('rev-notifications.csv', parse_dates=['created_date'])
tx = pd.read_csv('rev-transactions.csv', parse_dates=['created_date'])

# Règles fixées en partie 1
WINDOW = pd.Timedelta(days=60)
MIN_TENURE = pd.Timedelta(days=60)
CUTOFFS = {'fit': pd.Timestamp('2019-01-15'),
           'validation': pd.Timestamp('2019-02-15'),
           'test': pd.Timestamp('2019-03-15')}
X_OFFRE, BUDGET = 5, 500          # € par offre, offres par mois (hypothèses)
INTERCHANGE, USD_TO_EUR = 0.003, 0.89

## 1. Audit des données

| Constat | Décision |
|---|---|
| 202 transactions > 100 000 $, jusqu'à 85 milliards $ (virements) | erreurs de saisie : montants **plafonnés** au 99e percentile, calculé sur la période avant la 1re coupure |
| 58 604 transactions à 0 $ | paiements annulés, en attente ou refusés (REVERTED, PENDING, DECLINED) : gardés comme **signal d'échec**, sans effet sur les montants |
| Infos marchand vides à 42 % | vides seulement hors paiements carte (complètes à 100 % pour les paiements carte) : utilisées sur les paiements carte |
| Opt-in marketing inconnus pour 34 % des clients | inconnu ≠ non : on ajoute une colonne « inconnu » |
| `num_referrals`, `num_successful_referrals` toujours à 0 | retirées |
| 664 clients sans aucune transaction | jamais actifs, donc jamais dans le tableau |
| 715 notifications datées avant l'inscription | une par client, moins de 19 h avant, notifications d'onboarding : décalage d'horloge, gardées |
| `plan` | **fuite** (valeur au jour de l'export) : retirée |

In [ ]:
# Vérifications de l'audit
print('Transactions > 100 000 $ :', (tx.amount_usd > 1e5).sum())
print('Transactions à 0 $ :', (tx.amount_usd == 0).sum())
print(tx[tx.amount_usd == 0].transactions_state.value_counts().to_dict())
print('Infos marchand vides (paiements carte) :',
      f"{tx[tx.transactions_type == 'CARD_PAYMENT'].ea_merchant_mcc.isna().mean():.1%}")
print('Opt-in inconnus :', f"{users.attributes_notifications_marketing_push.isna().mean():.1%}")
print('Colonnes constantes :', [c for c in users.columns if users[c].nunique() <= 1])
print('Clients sans transaction :', (~users.user_id.isin(tx.user_id)).sum())

# Plafond des montants : 99e percentile, calculé uniquement sur le passé de la 1re coupure
CAP = tx.loc[(tx.created_date < CUTOFFS['fit']) & (tx.amount_usd > 0), 'amount_usd'].quantile(0.99)
tx['montant'] = tx.amount_usd.clip(upper=CAP)
print(f'Plafond des montants : {CAP:.0f} $')

## 2. `build_dataset(cutoff)` : une ligne par client actif

Toutes les features utilisent **uniquement** ce qui s'est passé avant la coupure ; le label uniquement ce qui se passe après.

In [ ]:
ECHEC = ['DECLINED', 'FAILED', 'REVERTED', 'CANCELLED']
TOP_PAYS = users.country.value_counts().index[:5]

def build_dataset(cutoff):
    passe = tx[tx.created_date < cutoff]
    futur = tx[(tx.created_date >= cutoff) & (tx.created_date < cutoff + WINDOW)]
    p90 = passe[passe.created_date >= cutoff - pd.Timedelta(days=90)]
    p30 = passe[passe.created_date >= cutoff - pd.Timedelta(days=30)]
    ok90 = p90[p90.transactions_state == 'COMPLETED']
    carte90 = ok90[ok90.transactions_type == 'CARD_PAYMENT']

    # clients actifs (une transaction dans les 60 jours avant) et ancienneté >= 60 jours
    actifs = passe[passe.created_date >= cutoff - WINDOW].user_id.unique()
    u = users.set_index('user_id').loc[actifs]
    d = pd.DataFrame(index=u.index)
    d['anciennete_j'] = (cutoff - u.created_date).dt.days
    d = d[d.anciennete_j >= MIN_TENURE.days]
    idx = d.index
    par_client = lambda s: s.reindex(idx).fillna(0)

    # Activité
    d['recence_j'] = (cutoff - passe.groupby('user_id').created_date.max().reindex(idx)).dt.days
    d['nb_tx_30j'] = par_client(p30.groupby('user_id').size())
    d['nb_tx_90j'] = par_client(p90.groupby('user_id').size())
    d['tendance'] = d.nb_tx_30j / (d.nb_tx_90j / 3 + 1)

    # Montants (transactions terminées, plafonnées)
    d['montant_90j'] = par_client(ok90.groupby('user_id').montant.sum())
    d['panier_moyen_90j'] = par_client(ok90.groupby('user_id').montant.mean())

    # Types de transactions (part sur 90 jours)
    for t in ['CARD_PAYMENT', 'TRANSFER', 'TOPUP', 'EXCHANGE', 'ATM']:
        d[f'part_{t.lower()}'] = par_client(p90[p90.transactions_type == t].groupby('user_id').size()) / d.nb_tx_90j.clip(lower=1)

    # Échecs
    d['taux_echec_90j'] = par_client(p90[p90.transactions_state.isin(ECHEC)].groupby('user_id').size()) / d.nb_tx_90j.clip(lower=1)

    # Marchands (paiements carte uniquement)
    d['nb_pays_marchands_90j'] = par_client(carte90.groupby('user_id').ea_merchant_country.nunique())
    d['nb_categories_90j'] = par_client(carte90.groupby('user_id').ea_merchant_mcc.nunique())

    # Notifications reçues sur 90 jours
    n90 = notifs[(notifs.created_date < cutoff) & (notifs.created_date >= cutoff - pd.Timedelta(days=90))]
    d['nb_notifs_90j'] = par_client(n90.groupby('user_id').size())
    d['taux_echec_notifs_90j'] = par_client(n90[n90.status == 'FAILED'].groupby('user_id').size()) / d.nb_notifs_90j.clip(lower=1)

    # Profil
    d['age'] = cutoff.year - u.birth_year.reindex(idx)
    d['crypto'] = u.user_settings_crypto_unlocked.reindex(idx)
    d['nb_contacts'] = u.num_contacts.reindex(idx)
    for canal in ['push', 'email']:
        col = u[f'attributes_notifications_marketing_{canal}'].reindex(idx)
        d[f'optin_{canal}'] = col.fillna(0)
        d[f'optin_{canal}_inconnu'] = col.isna().astype(int)
    pays = u.country.reindex(idx).where(u.country.reindex(idx).isin(TOP_PAYS), 'autre')
    d = d.join(pd.get_dummies(pays, prefix='pays', dtype=int))

    # Valeur du client (€ par an) : frais + commission carte, sur 90 jours, annualisée
    frais = par_client(ok90[ok90.transactions_type == 'FEE'].groupby('user_id').montant.sum())
    carte = par_client(carte90.groupby('user_id').montant.sum())
    d['valeur_an_eur'] = (frais + INTERCHANGE * carte) * 4 * USD_TO_EUR

    # Label
    d['churn'] = (~d.index.isin(futur.user_id)).astype(int)
    return d.reset_index()

fit = build_dataset(CUTOFFS['fit'])
val = build_dataset(CUTOFFS['validation'])
print(f'fit : {len(fit)} clients, churn {fit.churn.mean():.1%}, {fit.shape[1] - 2} colonnes')
print(f'validation : {len(val)} clients, churn {val.churn.mean():.1%}')

## 3. Tableau anti-fuite : « le jour de la coupure, est-ce que je connais cette valeur ? »

| Colonne(s) | Connue à la coupure ? | Décision |
|---|---|---|
| activité, montants, types, échecs, marchands, notifications | oui : calculées uniquement sur les transactions avant la coupure | gardées |
| ancienneté, âge, pays | oui : ne changent pas (ou peu) | gardées |
| `crypto`, `nb_contacts`, opt-in | **partiellement** : photo prise à l'export (mai 2019), pas à la coupure | gardées, **limite à signaler** |
| `plan` | **non** : plan au jour de l'export, peut avoir changé à cause du départ | **retirée** |
| `num_referrals`, `num_successful_referrals` | toujours 0 | retirées |
| `valeur_an_eur` | oui : calculée sur les 90 jours avant | utilisée pour la décision, pas comme feature du modèle |

## 4. Qu'est-ce qui va avec le churn ?

Pour chaque feature, sur les clients du **fit** : taux de churn chez les 25 % les plus bas et les 25 % les plus hauts (0 contre 1 pour les colonnes oui/non).

⚠️ Ce tableau montre ce qui **va avec** le churn, pas ce qui le **cause**.

In [ ]:
FEATURES = [c for c in fit.columns if c not in ['user_id', 'churn', 'valeur_an_eur']]

rows = []
for f in FEATURES:
    x = fit[f]
    if x.nunique() <= 2:
        bas, haut = fit[x == x.min()].churn.mean(), fit[x == x.max()].churn.mean()
        lb, lh = f'= {x.min():g}', f'= {x.max():g}'
    else:
        q1, q3 = x.quantile([0.25, 0.75])
        if q1 == q3:   # trop de valeurs identiques : 0 contre > 0
            bas, haut, lb, lh = fit[x <= q1].churn.mean(), fit[x > q1].churn.mean(), f'≤ {q1:g}', f'> {q1:g}'
        else:
            bas, haut, lb, lh = fit[x <= q1].churn.mean(), fit[x >= q3].churn.mean(), f'≤ {q1:.3g}', f'≥ {q3:.3g}'
    rows.append({'feature': f, 'bas': lb, 'churn si bas': f'{bas:.0%}',
                 'haut': lh, 'churn si haut': f'{haut:.0%}', 'écart (points)': round(100 * (haut - bas))})

lecture = pd.DataFrame(rows).sort_values('écart (points)', key=abs, ascending=False)
print(f'Taux de churn moyen (fit) : {fit.churn.mean():.0%}')
lecture

### Lecture (brouillon, à reformuler avec nos mots)

Taux de churn moyen : 14 %.

1. **Le désengagement est le signal n°1.** Rien depuis 18 jours ou plus : 36 % de churn, contre 2 % pour ceux qui ont utilisé leur compte la veille. Même chose pour peu de transactions sur 30 et 90 jours, et pour une activité qui ralentit (tendance basse : 31 %).
2. **Un usage étroit, c'est un usage fragile.** 1 seule catégorie de commerce : 29 % de churn, contre 3 % au-delà de 13 catégories. 1 seul pays marchand : 23 %, contre 4 % au-delà de 3. Revolut utilisé comme compte principal (beaucoup de catégories, voyages) retient bien mieux qu'un usage d'appoint.
3. **Peu de liens sociaux, plus de départs.** 0 contact : 24 %, contre 6 % au-delà de 24 contacts.
4. **Le pays compte un peu.** Royaume-Uni : 18 % ; Irlande et Roumanie : 6 %.
5. **Ce qui compte peu :** taux d'échec des paiements (+3 points), âge, notifications, opt-in marketing.

⚠️ Ce sont des associations, pas des causes. Exemple : un client qui part arrête d'utiliser sa carte, donc il a peu de catégories. Le petit nombre de catégories est un **symptôme**, pas forcément une raison du départ.

## 5. Modèles : 5 features simples contre toutes les features (validation)

In [ ]:
SIMPLES = ['anciennete_j', 'recence_j', 'nb_tx_30j', 'nb_tx_90j', 'tendance']

def entraine(cols):
    logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    trees = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, random_state=0)
    return {f'logistique ({len(cols)} f.)': logit.fit(fit[cols], fit.churn).predict_proba(val[cols])[:, 1],
            f'arbres ({len(cols)} f.)': trees.fit(fit[cols], fit.churn).predict_proba(val[cols])[:, 1]}

scores = {'baseline (récence)': val.recence_j}
scores.update(entraine(SIMPLES))
scores.update(entraine(FEATURES))

## 6. Décision en euros : classer par risque, ou par perte attendue ?

- **Classement par risque** : on cible les 500 clients avec le `p` le plus haut.
- **Classement par perte attendue** : on cible les 500 clients avec le `p × valeur_an_eur` le plus haut (les gros clients risqués d'abord).

Gain = Σ (churners ciblés × Z × leur valeur) − 500 × X. **Z est inconnu** : on montre 30 % et 50 %.

In [ ]:
def gain(cibles, z):
    return (cibles.churn * z * cibles.valeur_an_eur).sum() - len(cibles) * X_OFFRE

rows = []
for nom, s in scores.items():
    v = val.assign(p=s)
    for critere, cle in [('risque', v.p), ('perte attendue', v.p * v.valeur_an_eur)]:
        if nom.startswith('baseline') and critere == 'perte attendue':
            continue
        cibles = v.loc[cle.nlargest(BUDGET).index]
        rows.append({'méthode': nom, 'classement': critere,
                     'PR-AUC': round(average_precision_score(val.churn, s), 3),
                     'churners ciblés': int(cibles.churn.sum()),
                     'gain € Z=30%': round(gain(cibles, 0.30)),
                     'gain € Z=50%': round(gain(cibles, 0.50))})
pd.DataFrame(rows)

### Lecture (brouillon)

1. **Plus de features, meilleur ciblage.** Avec 30 features, les arbres passent de 252 à 282 churners dans le top 500 (PR-AUC 0,389 → 0,436). Ils battent la baseline (214) et la régression logistique.
2. **Ceux qui partent sont ceux qui rapportent le moins.** Classés par risque seul, les 500 clients ciblés valent si peu que l'offre à 5 € perd plus de 2 000 € par mois, même si elle retient la moitié des clients.
3. **Classer par perte attendue (`p × valeur`) change tout.** On cible moins de churners (environ 88), mais des clients qui valent beaucoup plus. Le gain approche zéro à Z = 30 % et devient positif à Z = 50 % (+435 à +518 €).
4. **Prochaine étape :** ne plus fixer 500 offres, mais envoyer seulement quand `p × Z × valeur > X`, et chercher le prix d'offre maximum rentable.